# Mizal Analytics: Model Development Notebook

**Project:** Predicting High-Suitability Team Members for Task Assignment  
**Student:** Kaye Ashley B. Mizal  
**System:** Mizal Analytics: Elite Talent Predictor

This notebook documents the model-development workflow used for the final predictive analytics system. It covers dataset loading, preprocessing, feature engineering, model training, model comparison, final evaluation, and feature importance interpretation.

## 1. Import Libraries

The project uses pandas and NumPy for data handling, scikit-learn for model development and evaluation, and matplotlib for visualizations. The final deployed app uses Streamlit, but the model-development workflow is shown here in notebook form.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42
DATA_FILE = "structured_data (1).csv"

## 2. Load Dataset

The local dataset is based on Kaggle's Employee Performance Evaluation Dataset. The working file contains employee performance indicators used by the Mizal Analytics system.

In [ ]:
df = pd.read_csv(DATA_FILE)
print("Dataset shape:", df.shape)
df.head()

## 3. Dataset Understanding

This step checks the available columns, missing values, duplicate employee records, and basic descriptive statistics. These checks are important because data quality directly affects model reliability.

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values per column:")
print(df.isna().sum())

if "employee_id" in df.columns:
    print("\nDuplicate employee IDs:", df["employee_id"].duplicated().sum())

print("\nPerformance rating distribution:")
print(df["performance_rating"].value_counts(dropna=False))

df.describe(include="all")

## 4. Define Predictive Features

The final system uses six core performance indicators. Hours worked is intentionally excluded to avoid rewarding overwork, and innovation score is retained only as metadata because it may be more subjective than the other indicators.

In [ ]:
features = [
    "average_task_quality",
    "tasks_completed",
    "projects_led",
    "deadline_met_score",
    "client_satisfaction_score",
    "efficiency_score"
]

features

## 5. Data Cleaning and Preprocessing

The preprocessing pipeline follows the same logic used in the Streamlit application. It removes duplicate employee records, coerces predictive features to numeric values, fills missing values with medians, and clips extreme values between the 1st and 99th percentiles.

In [ ]:
clean_df = df.copy()

if "employee_id" in clean_df.columns:
    clean_df = clean_df.drop_duplicates(subset=["employee_id"], keep="last")
else:
    clean_df = clean_df.drop_duplicates()

for col in features:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")
    median_value = clean_df[col].median()
    if pd.isna(median_value):
        median_value = 0
    clean_df[col] = clean_df[col].fillna(median_value)

    lower_limit = clean_df[col].quantile(0.01)
    upper_limit = clean_df[col].quantile(0.99)
    if not pd.isna(lower_limit) and not pd.isna(upper_limit):
        clean_df[col] = clean_df[col].clip(lower=lower_limit, upper=upper_limit)

print("Cleaned dataset shape:", clean_df.shape)
clean_df[features].describe().round(2)

## 6. Feature Engineering: Composite Score and Target Variable

The original `performance_rating` column is not used as the final target because it represents a general rating and is highly imbalanced. Instead, the project creates a business-specific target named `is_high_suitability`. Employees whose composite score reaches the top 33% cutoff are labeled High Suitability.

In [ ]:
clean_df["composite_score"] = (
    clean_df["average_task_quality"] * 0.30 +
    clean_df["tasks_completed"] * 0.20 +
    clean_df["projects_led"] * 0.15 +
    clean_df["deadline_met_score"] * 0.15 +
    clean_df["client_satisfaction_score"] * 0.10 +
    clean_df["efficiency_score"] * 0.10
)

threshold = clean_df["composite_score"].quantile(0.67)
clean_df["is_high_suitability"] = (clean_df["composite_score"] >= threshold).astype(int)

print(f"Elite cutoff score: {threshold:.2f}")
print(clean_df["is_high_suitability"].value_counts().rename({0: "Standard", 1: "High Suitability"}))

## 7. Target Distribution Visualization

The chart below shows the number of employees classified as Standard Suitability and High Suitability after target engineering.

In [ ]:
class_counts = clean_df["is_high_suitability"].value_counts().sort_index()
labels = ["Standard", "High Suitability"]

plt.figure(figsize=(6, 4))
plt.bar(labels, class_counts.values, color=["#4B8BBE", "#2EAD63"])
plt.title("Target Variable Distribution")
plt.ylabel("Number of Employees")
for i, value in enumerate(class_counts.values):
    plt.text(i, value + 30, str(value), ha="center")
plt.show()

## 8. Train-Test Split

The dataset is split using stratification so the Standard and High Suitability classes remain proportionally represented in both training and testing sets.

In [ ]:
X = clean_df[features]
y = clean_df["is_high_suitability"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## 9. Human-Implemented Model: Decision Tree

The Decision Tree Classifier is used as the human-implemented baseline because it is interpretable and easy to explain in a business setting.

In [ ]:
decision_tree = DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=8,
    min_samples_leaf=4,
    random_state=RANDOM_STATE
)

decision_tree.fit(X_train, y_train)
dt_predictions = decision_tree.predict(X_test)

print(classification_report(y_test, dt_predictions, target_names=["Standard", "High Suitability"]))

## 10. AI-Recommended Model: Random Forest

The Random Forest Classifier is used as the AI-recommended and final selected model because it combines multiple trees, improves stability, and reduces overfitting compared with a single Decision Tree.

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=8,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    oob_score=True
)

random_forest.fit(X_train, y_train)
rf_predictions = random_forest.predict(X_test)

print(classification_report(y_test, rf_predictions, target_names=["Standard", "High Suitability"]))
print(f"Random Forest OOB score on training split: {random_forest.oob_score_:.4f}")

## 11. Model Comparison

The table below compares the Decision Tree baseline and Random Forest model using common classification metrics. The Random Forest has stronger overall performance and is therefore selected as the final model.

In [ ]:
def summarize_model(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Weighted Precision": precision_score(y_true, y_pred, average="weighted"),
        "Weighted Recall": recall_score(y_true, y_pred, average="weighted"),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted"),
        "High Suitability Precision": precision_score(y_true, y_pred, pos_label=1),
        "High Suitability Recall": recall_score(y_true, y_pred, pos_label=1),
        "High Suitability F1": f1_score(y_true, y_pred, pos_label=1),
    }

comparison = pd.DataFrame([
    summarize_model("Decision Tree", y_test, dt_predictions),
    summarize_model("Random Forest", y_test, rf_predictions)
])

comparison.round(4)

## 12. Cross-Validation

Five-fold stratified cross-validation provides another view of model stability. This helps confirm that the model performance is not only due to one train-test split.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

dt_cv_scores = cross_val_score(decision_tree, X, y, cv=cv, scoring="accuracy")
rf_cv_scores = cross_val_score(random_forest, X, y, cv=cv, scoring="accuracy")

print(f"Decision Tree CV Accuracy: {dt_cv_scores.mean():.4f} +/- {dt_cv_scores.std():.4f}")
print(f"Random Forest CV Accuracy: {rf_cv_scores.mean():.4f} +/- {rf_cv_scores.std():.4f}")

## 13. Final Model Evaluation Using Full Dataset OOB Validation

The deployed Streamlit system trains the final Random Forest model on the full cleaned dataset and uses out-of-bag evaluation. This matches the final documentation metrics.

In [ ]:
final_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=8,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    oob_score=True
)

final_model.fit(X, y)
oob_predictions = np.argmax(final_model.oob_decision_function_, axis=1)
final_cm = confusion_matrix(y, oob_predictions)

print(f"Final Random Forest OOB Accuracy: {final_model.oob_score_:.4f}")
print("Confusion Matrix:")
print(final_cm)

ConfusionMatrixDisplay(final_cm, display_labels=["Standard", "High Suitability"]).plot(cmap="Blues")
plt.title("Final Random Forest OOB Confusion Matrix")
plt.show()

## 14. Feature Importance

Feature importance helps explain which employee indicators most influenced the final Random Forest prediction. In the current dataset, tasks completed and client satisfaction score are the strongest signals.

In [ ]:
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": final_model.feature_importances_
}).sort_values("Importance", ascending=False)

importance_df

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.barh(importance_df["Feature"], importance_df["Importance"], color="#2EAD63")
plt.gca().invert_yaxis()
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.show()

## 15. Final Interpretation

The Random Forest model was selected as the final predictive model because it outperformed the Decision Tree baseline and achieved strong out-of-bag accuracy. The final model supports the business objective by identifying employees who are more suitable for high-priority task assignments based on measurable performance indicators. The model should still be used as decision support, with managers reviewing final task assignment decisions using business context such as availability, workload, and specialization.